# Homework 3

# CS328 — Numerical Methods for Visual Computing and Machine Learning
- - -


<div style="text-align: center">
<br>
<b>This assignment will be graded automatically through our grading tool.</b>
<br><br>
<b>Due: 26.11.2025 at 21:00 (UTC+1) </b> <br><br> </div>
</div>

This notebook contains literate code, i.e. brief fragments of Python surrounded by descriptive text. Please complete/extend this notebook for your homework submission:

* Only write your solution code in the areas that start with "`# YOUR CODE HERE`". Please do not change the variable names that are set as ellipsis `(...)`, but replace the ellipses with the right definition and indicate your solution using these variables. You can add more cells to test your answer.
  
* We are using cell tags to prevent you from editing or deleting cells that are required for our grading tool. If you are using Jupyter Lab / Notebook, you do not need to worry about this, as long as you do not try to delete/change the content of cells that Jupyter Lab/Notebook does not let you edit by default. If you are using some other IDE (VS Code, PyCharm, etc), **you have full responsibility to make sure that the cell tags are not modified, and that you do not change the code of cells with tag `"editable": false`** . Especially, you should not delete any of the existing cells from the Notebook. 
  
* Before handing in, **please make sure that your notebook runs from top to bottom after selecting "Kernel->Restart & Run All"** without causing any errors. If an exception raises during the correction, zero points will be assigned to the question.

Make sure to use the reference Python distribution so that project files can be opened by the TAs. In this course, we use <a href="https://www.anaconda.com/products/individual">Anaconda</a>, specifically the version based on Python 3.13.

<div class="alert alert-warning">
Homework assignments in CS328 count towards your final grade and therefore must be done individually.
</div>

> © Realistic Graphics Lab - EPFL (2025). This notebook is property of the Realistic Graphics Lab at EPFL and may not be redistributed. Posting of notebooks and/or solutions on the web (e.g. on GitHub) is not permitted.

$$
\newcommand{\vp}{\mathbf{p}}
\newcommand{\vl}{\mathbf{l}}
\newcommand{\vx}{\mathbf{x}}
\newcommand{\vxp}{\mathbf{x'}}
\newcommand{\va}{\mathbf{a}}
\newcommand{\vap}{\mathbf{a'}}
\newcommand{\vn}{\mathbf{n}}
\newcommand{\mA}{\mathbf{A}}
\newcommand{\mM}{\mathbf{M}}
\newcommand{\vb}{\mathbf{b}}
\newcommand{\vc}{\mathbf{c}}
\newcommand{\vx}{\mathbf{x}}
\newcommand{\vu}{\mathbf{u}}
\newcommand{\vv}{\mathbf{v}}
\newcommand{\mA}{\mathbf{A}}
\newcommand{\mL}{\mathbf{L}}
\newcommand{\mU}{\mathbf{U}}
\newcommand{\mV}{\mathbf{V}}
\newcommand{\mP}{\mathbf{P}}
\newcommand{\mI}{\mathbf{I}}
\newcommand{\mLamb}{\mathbf{\Lambda}}
$$


# Installing additional prerequisites

In this assignment, we will make use a library called **[bqplot](https://github.com/bloomberg/bqplot)** developed by Bloomberg, which enables fully interactive plots within the Jupyter notebook. By default, this library is not installed on the Anaconda Python distribution we're using in this course, hence you will need to install it first. To do so, simply run the cell below:

In [1]:
%pip install bqplot

Note: you may need to restart the kernel to use updated packages.


Afterwards, you will have to **restart** Jupyter notebook for the change to become effective. You can enter and run the following commands in a new cell to check if bqplot was installed correctly—they should display a figure with a pie chart.

```python
import bqplot as bqp
bqp.Figure(marks=[bqp.Pie(sizes=range(1, 6))], title='A pie chart!')
```

# Prelude

We begin by importing essential NumPy/SciPy/Matplotlib components that are needed to complete the exercises. The package ``scipy.optimize`` is new -- we'll use it for the last portion of this homework. The ``ipywidgets`` package is  used internally by some of the code snippets provided by us.

In [2]:
import numpy as np
import scipy.linalg as la

# New: Optimization package
import scipy.optimize as opt

# bqplot plotting library
import bqplot as bqp

# Import graphical user interface components used below
from ipywidgets import interact
from ipywidgets import FloatSlider, VBox

# Inverse Kinematics using Newton's method and the Pseudoinverse
$$
\newcommand{\vp}{\mathbf{p}}
\newcommand{\vx}{\mathbf{x}}
\newcommand{\vf}{\mathbf{f}}
\newcommand{\mA}{\mathbf{A}}
$$

$$
\newcommand{\vp}{\mathbf{p}}
\newcommand{\vx}{\mathbf{x}}
\newcommand{\vf}{\mathbf{f}}
\newcommand{\mA}{\mathbf{A}}
$$
Lifelike computer-animated characters and animals are increasingly pervasive in today’s society:  they are now commonly encountered in games, advertisements, feature animation, and a variety of other fields. It’s interesting to realize that all of these animated characters initially start out as static 3D shapes, not unlike stone sculptures that are unable to move. Before a character built in this way can be seen in motion, the precise way in which its shape can change over time must be characterized mathematically. The most common way of doing this entails designing a customized 3D skeleton that is then attached to its outer skin. Subsequently, any adjustment to the posture of the skeleton will result in a corresponding change to the posture of the character. The figure below shows a simple character with an embedded skeleton and two example poses that were created by  rotating the joints of the skeleton.

<br><img width="490" src="//rgl.s3.eu-central-1.amazonaws.com/media/uploads/wjakob/2016/12/06/ik-system.jpg"><br>

A skeleton can range from a few elements to massively complex arrangements that reproduce the entire biological structure of the person or animal in question. In either case, the skeleton consists only of repeated instances of one basic component: a *bone*. Fortunately, computer graphics bones are much simpler than actual bones in real life.

<br><img width="490" src="//rgl.s3.eu-central-1.amazonaws.com/media/uploads/wjakob/2016/12/06/inverse-kinematics-05.png"><br>

As you can see in the above figure, each bone is essentially a line segment with a joint where it is connected to the previous bone. The bone can rotate in any way, but the connection between two bones can never move apart. Each bone also has a fixed length, in other words: it is rigid and never compresses or expands. On the other side, the next bone is generally attached. In 3D, a variety of rotations are possible (e.g. around the X, Y, or Z axis). In two dimensions things are simpler, and a single angle is enough to completely characterize the rotation of a bone relative to its predecessor bone. Everything in this homework will be in two dimensions to keep things simple.

## Part 1: Forward Kinematics (35 points)

In this exercise, we'll investigate the mathematics of a very simple kind of "skeleton": a chain of bones in two dimensions with joint positions $\vp_i = (x_i, y_i)$. The first joint is rigidly attached to the origin (i.e. $\vp_0 = (0, 0)$) while the other joints and bones are free to move in any way. For simplicitly, we'll also assume that all of the bones have the same length $l_1=l_2=\ldots=1$.

<img width="490" src="//rgl.s3.eu-central-1.amazonaws.com/media/uploads/wjakob/2016/12/06/inverse-kinematics-01.png"> 

Each parameter $\theta_i\in[0,2\pi]$ specifies the counter-clockwise angle that the associated bone from joint $\vp_{i-1}$ to joint $\vp_i$ makes with its predecessor bone (the pair of bones are parallel if $\theta_i=0$). The first bone doesn't have a predecessor, hence $\theta_1$ is measured relative to the $X$ axis. Note how the complete set of bone angles $\theta_1, \theta_2, \ldots$ is all the information we need to compute the precise positions of all the joint positions in Euclidean space.

Forward kinematics (FK) is defined as the problem of converting a set of bone angles $\theta_i$ into joint positions $\vp_i$. Since $\vp_i$ depends on all of the preceding angles, we can think of each joint position as a function $\vp_i=\vp(\theta_1,\ldots,\theta_{i})$ 

**TODO** (15 pts): Your first task is to create a function ``chain_simple``, which solves the forward kinematics for a chain with at most one bone. The function should take an array of angles (in radians) as a parameter, which can be of length 0 or 1 (use the Python ``len()`` function to query the length of an array). When no angles are specified, the function should return the position of the first joint $(x_0,y_0)=(0, 0)$ as an 1D NumPy array. When a single angle is specified, it should return the position $x_1, y_1$.

Ensure that your implementation returns the expected example values in the cell below (up to minor rounding errors).

In [3]:
def chain_simple(theta):
    """
    This function computes the position of the end point for a single-bone skeleton.

    Parameters:
    -----------
    theta: numpy.ndarray
        An array of input angle theta, containing 0 or 1 value.
    
    Returns:
    --------
    position: numpy.ndarray of shape (2,)
        An array containing the coordinate of the joint. 
        When no angles are specified, i.e input is [], the function should return 
        the position of the (x0, y0) as an 1D NumPy array. When a single angle 
        is specified, it should return the position (x1, y1).
    """
   
    if len(theta) == 0:
         position = np.array([0., 0.])
    elif len(theta) == 1:
        position =  np.array([np.cos(theta[0]), np.sin(theta[0])])

    return position


In [4]:
assert chain_simple([]).shape == (2,)
assert np.allclose(chain_simple([]), np.array([0., 0.]))
assert np.allclose(chain_simple([0.]), np.array([1., 0.]))



### 1.1 Helper function

We provide the function ``fk_demo()`` below to interactively explore the possible chain configurations via forward kinematics. The implementation uses the ``bqplot`` library mentioned above and is fairly technical. You are welcome but not expected to read or understand how it works.

In [5]:
def fk_demo(chain_func, theta, extra = [[],[]]):
    '''
    This function visualizes the configuration of a chain of bones
    and permits interactive changes to its state. It expects two arguments:
    
    ``chain_func``: a function that implements forward kinematics by
    turning a sequence of angles (theta_1, theta_2, ..., theta_n) into
    the position of the last joint of this chain (x_n, y_n).
    
    ``theta``: an array with the initial angles of all joints
    
    ``extra``: An optional argument which can be used to plot
    additional points that are highlighted in red
    '''
    
    # Function which repeatedly calls ``chain_func`` to compute all joint positions
    def chain_all(theta):
        return np.column_stack([chain_func(theta[:i]) for i in range(0, len(theta) + 1)])

    # Determine size and initial configuration
    size = len(theta)
    positions = chain_all(theta)

    # Define the range of the plotting frame
    scales = { 'x': bqp.LinearScale(min=-size-1, max=size+1),
               'y': bqp.LinearScale(min=-size-1, max=size+1) }

    # Create a scatter plot (for joints), a line plot (for bones), and
    # another scatter plot (to draw extra points specified the ``extra`` argument)
    scat  = bqp.Scatter(scales=scales)
    lines = bqp.Lines(scales=scales)
    scat2 = bqp.Scatter(scales=scales, colors=['red'])

    # Create a figure that combines the three plots
    figure = bqp.Figure(marks=[scat, scat2, lines])
    figure.layout.height = '500px'
    figure.layout.width = '500px'

    # Initialize the plots with the initial data
    scat.x, scat.y = positions
    lines.x, lines.y = positions
    scat2.x, scat2.y = extra
    
    sliders = []
    
    # For each angle theta_i,
    for i in range(len(theta)):
        # Create a graphical slider
        slider = FloatSlider(min=0, max=2*np.pi, value=theta[i], step=1e-3)
        
        # Define a callback function that will be triggered when the slider is moved
        def callback(value, i = i):
            theta[i] = value['new']
            positions = chain_all(theta)
            scat.x, scat.y = positions
            lines.x, lines.y = positions

        # "Attach" the callback function to the slider
        slider.observe(callback, 'value')
        sliders.append(slider)

    # Combine the plots and sliders in a vertical arrangement
    return VBox([*sliders, figure])

### 1.2 Visualization of the forward kinematics

**TODO (0pts)**: To ensure that your implementation of ``chain_simple`` satisfies all the specifications, run the following cell that invokes the ``fk_demo()`` function with arguments ``chain_simple`` and ``[0.]`` (the initial parameters of a flat chain). You should be able to drag a slider from 0 to $2\pi$ and see a visual representation of a 1-bone chain turning counter-clockwise.

In [6]:
fk_demo(chain_simple, [0])

### 1.3 Longer chains

**TODO** (15 pts): Create a function ``chain``, which solves the forward kinematics for an arbitrarily long sequence of bones. The function should take an arbitrary-length array of angles as a parameter. When no angles are specified, the function should return the position $(x_0, y_0)$ as before. When $i$ angles are specified, it should (only) return  the joint position $(x_{i}, y_{i})$.

Ensure that your implementation returns the expected example values in the cell below (up to minor rounding errors).

In [7]:
def chain(thetas):
    """
    This function computes the position of the end point for a skeleton with an 
    arbitrarily long sequence of bones.

    Parameters:
    -----------
    thetas: numpy.ndarray
        An array of input angle theta, containing >=0 value.
    
    Returns:
    --------
    position: numpy.ndarray of shape (2,)
        An array containing the coordinate of the joint. 
        When no angles are specified, i.e input is [], the function should return 
        the position of the (x0, y0). When a single angle is 
        specified, it should return the position (x1, y1) as an 1D NumPy array.
    """
    length = len(thetas)
    position = np.zeros(2)
    sum = 0
    for i in range(length):
        sum += thetas[i]
        position += np.array([np.cos(sum), np.sin( sum)])
       
    return position


In [8]:
assert chain([np.pi, np.pi, np.pi, np.pi]).shape == (2,)
assert np.allclose(chain([np.pi, np.pi, np.pi, np.pi]), np.array([0, 0]))


### 1.4 Attempting to reach a certain position

Run the command ``fk_demo(chain, [0, 0, 0, 0, 0], [[-2], [3]])`` below. You should see a chain with five segments and five corresponding sliders, as well as an additional point highlighted in red.

**TODO (5 points)**: Find a configuration of angles that brings the endpoint of the chain as close as possible to the highlighted location ``[-2, 3]]``. An exact match is not necessary, but the points should overlap by a significant margin. Write the parameters you found in the `angles` variable in the next cell.

In [9]:
# TODO:
fk_demo(chain, [0, 0, 0, 0, 0], [[-2], [3]])

In [10]:
angles = [1.52, 0, 0.74, 0, 1.46]

In [11]:
assert len(angles) == 5

## Part 2: Inverse Kinematics (45 pts)

Problems similar to the one in Section 1.4 are tedious to solve by hand: all of the parameters are interdependent and must be adjusted in a coordinated manner. So-called *inverse kinematics* techniques apply numerical root finding to determine solutions to this problem in an automated way. Most modern animation systems have builtin support for inverse kinematics since it allows for a much more convenient workflow: rather than having to tweak each individual bone, artists can directly specify a target shape, and the system will automatically infer all the necessary rotations.

In this part of the exercise, we will use inverse kinematics to automatically determine $\theta_1,\ldots,\theta_n$ such that

$$
\vp(\theta_1,\ldots,\theta_n) = \vp_{\mathrm{target}}
$$

for a given value $\vp_{\mathrm{target}}\in\mathbb{R}^2$. In other words: the user can move around the endpoint of the chain, and the skeleton will automatically reconfigure itself to follow. This is illustrated in the following figure:

<img width="900" src="//rgl.s3.eu-central-1.amazonaws.com/media/uploads/wjakob/2016/12/06/inverse-kinematics-04.png"> 

All good numerical root finding techniques require the ability to evaluate the Jacobian of $\vp$, i.e. all the partial derivatives $\frac{\partial\vp(\theta_1,\ldots,\theta_n)}{\partial \theta_j}$. The partial derivatives encode how a small perturbation of each of the angles $\theta_j$ leads to a corresponding change in $\vp(\theta_1,\ldots,\theta_n)$. As before, we'll first look at a 1-segment chain and then derive a solution for the general problem.

**TODO** (10 pts): Implement a function ``dchain_simple(theta)`` which takes an array with one entry, and which computes the function $\frac{\partial \vp(\theta_1)}{\partial \theta_1}$. The return value should be a two-dimensional array with one column and two rows containing the partial derivatives of the coordinate values $x_1$ and $y_1$. You should use analytic methods -- approximating the derivatives via finite differences is not allowed.

Ensure that your implementation returns the expected example values in the cell below (up to minor rounding errors).


In [12]:
def dchain_simple(theta):
    """
    This function computes the derivative of the position w.r.t. angle. 

    Parameters:
    -----------
    theta: numpy.ndarray
        An array of input angle theta, containing 1 value.
    
    Returns:
    --------
    derivative: numpy.ndarray of shape (2,1)
        An array containing the partial derivatives of the coordinate values x1 and y1.
    """
    # YOUR CODE HERE
    derivative = np.array([[-np.sin(theta[0])],[ np.cos(theta[0])]])
    return derivative


In [13]:
assert np.allclose(dchain_simple([0]), np.array([[0.], [1.]]))
assert dchain_simple([0]).shape == (2,1)


### 2.1 Implementing the full Jacobian function

Having finished the version for a single bone, we'll now turn to the full Jacobian $\nabla \vp(\theta_1, \ldots, \theta_n)$, which is a $2\times n$ matrix containing the partial derivatives with respect to all angles.

**TODO** (20 pts): Implement a function ``dchain(theta)`` which accepts an 1D array of angles with length $\ge 1$ and computes the Jacobian $\nabla \vp(\theta_1, \ldots, \theta_n)$, a $2\times n$ matrix.

Ensure that your implementation returns the expected example values in the cell below (up to minor rounding errors).

In [14]:
def dchain(thetas):
    """
    This function computes the full Jacobian matrix for the partial derivatives of position w.r.t all angles.

    Parameters:
    -----------
    thetas: numpy.ndarray
        An array of input angle theta, containing n>=1 value.
    
    Returns:
    --------
    derivative: numpy.ndarray of shape (2,n)
        An array containing the full Jacobian Matrix.
    """
    length = len(thetas)
    derivative = np.zeros((2, length))
    cumsum_thetas = np.cumsum(thetas)  
    for i in range(length):
        dx_dtheta_i = -np.sum(np.sin(cumsum_thetas[i:]))
        dy_dtheta_i =  np.sum(np.cos(cumsum_thetas[i:]))
        derivative[:, i] = [dx_dtheta_i, dy_dtheta_i]

    return derivative


In [15]:
assert np.allclose(dchain([0, 0, 0, 0]), np.array([[ 0.,  0.,  0.,  0.], [ 4.,  3.,  2.,  1.]]))
assert dchain([0, 0, 0, 0]).shape == (2,4)


### 2.2 Solving the inverse kinematics problem using Newton's Method

**TODO** (15 pts): Implement a function ``newton(theta, target)`` that takes a 1-dimensional array of angles as a starting guess as well as a 2D target position (also specified as a 1-dimensional array) as input. The implementation should perform a fixed 8 iterations of Newton's method to try to solve the equation $\vp(\theta_1,\ldots,\theta_n) = \vp_{\mathrm{target}}$ and return the final set of parameters $\theta_1,\ldots,\theta_n$ as an 1-dimensional NumPy array. You can use the function ``la.pinv`` to compute the pseudoinverse.

Ensure that your implementation returns the expected example values in the cell below (up to minor rounding errors).

In [16]:
def newton(thetas, target):
    """
    This function approximates the set of angles used in previous problem setting, with Newton's Method. 

    Parameters:
    -----------
    thetas: numpy.ndarray
        An array of provided starting guess for angle theta, containing n>=1 value.

    target: numpy.ndarray
        A 2-D target position. 
    
    Returns:
    --------
    thetas: numpy.ndarray of shape (n,)
        The final estimated angles array . 
    """
    # YOUR CODE HERE
    for i in range(8):
        thetas = thetas - np.linalg.pinv(dchain(thetas)) @ (chain(thetas) - target)
    return thetas


In [17]:
assert newton(np.array([0., 0.]), np.array([0., 1.])).shape == (2,)
assert np.allclose(chain(newton(np.array([0., 0.]), np.array([0.5, 0.5]))), np.array([0.5, 0.5]))

### 2.3 One more helper function

In [18]:
def ik_demo(solver, size):    
    theta = np.zeros(size, dtype=np.float64)
    
    # Function which repeatedly calls ``chain`` to compute all joint positions
    def chain_all(theta):
        return np.column_stack([chain(theta[:i]) for i in range(0, len(theta) + 1)])

    # Callback that is invoked when the user drags the red endpoint around
    def refresh(_):
        # 'theta' is a variable of the parent function, we want to modify it here
        nonlocal theta
        
        # Target position
        target = np.array([scat2.x[0], scat2.y[0]])
        
        # Don't try to solve the problem if the user dragged the point out of the circle
        if la.norm(target) > size:
            return
        
        # Call the provided IK solver
        theta = solver(theta, target)
        
        # Update the positions
        values = chain_all(theta)
        scat.x, scat.y = values
        lines.x, lines.y = values
    
    # Similar to fk_solver(), create a number of plots and merge them
    scales = { 'x': bqp.LinearScale(min=-size-1, max=size+1),
               'y': bqp.LinearScale(min=-size-1, max=size+1) }

    scat  = bqp.Scatter(scales=scales)
    lines = bqp.Lines(scales=scales)

    # Create a circle which marks the boundary of where the red point can be moved
    circle_x = np.cos(np.linspace(0, 2*np.pi, 100)) * size
    circle_y = np.sin(np.linspace(0, 2*np.pi, 100)) * size
    circle = bqp.Lines(x=circle_x, y=circle_y,
                       scales=scales, colors=['gray'])
    
    # Special plot, which contains the red endpoint that can be moved
    scat2 = bqp.Scatter(scales=scales,
                        enable_move=True, 
                        update_on_move=True,
                        colors=['red'])

    # Initialize the visualizations with the default configuration
    values = chain_all(theta)
    scat.x, scat.y = values
    lines.x, lines.y = values
    scat2.x, scat2.y = chain(theta).reshape(2, 1)
    
    # Call the 'refresh' function when the red dot is moved
    scat2.observe(refresh, names=['x', 'y'])

    figure = bqp.Figure(marks=[scat, scat2, lines, circle])
    figure.layout.height = '500px'
    figure.layout.width = '500px'
    return figure

We provide the function ``ik_demo()`` below to interactively explore the possible chain configurations via inverse kinematics. Similar to ``fk_demo()``, the function is fairly technical. You are welcome but not expected to read or understand how it works.

### 2.4 Putting everything together

Finally, let's visualize the behavior of the completed inverse kinematics solver. 

**TODO**:
1. Invoke the IK demonstration with with 4 segments, i.e. ``ik_demo(newton, 4)``. You should be able to move the red endpoint with your mouse cursor, leading to a smooth adjustment of the chain configuration.<br><br>

2. Invoke the IK demonstration with with 30 segments, i.e. ``ik_demo(newton, 30)``. Does the algorithm still work? Can you break it by moving the cursor too quickly? What happens in this case?

In [19]:
# Solution
ik_demo(newton, 4)

Figure(fig_margin={'top': 60, 'bottom': 60, 'left': 60, 'right': 60}, layout=Layout(height='500px', width='500…

In [20]:
ik_demo(newton, 30)

Figure(fig_margin={'top': 60, 'bottom': 60, 'left': 60, 'right': 60}, layout=Layout(height='500px', width='500…

## 3 Regularized Inverse Kinematics (10 points)

The inverse kinematics solution from Example 2 provides generally reasonable results by attempting to make the smallest change to the bone angles that will allow the chain to reach a particular point. However, in some cases there might be additional requirements we'd like the solution to satisfy. For instance, it might be unnatural for the character to bend a joint more than a few degrees. In this case, we could try to find a solution to an optimization problem that compromises between reaching the target position and bending joints by an overly large amount.

$$
\DeclareMathOperator*{\argmin}{argmin}
\argmin_{\theta_1,\ldots,\theta_n} \alpha\cdot\|\vp(\theta_1,\ldots,\theta_n)-\vp_\mathrm{target}\|_2^2 + \sum_{i=1}^n\theta_i^2
$$

To optimize such a function, we will use SciPy's ``opt.minimize`` function (see the docs [here](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html)). It takes as argument the objective function to be minimized, as well as the input parameters and the target to reach.

**TODO**: (5 pts) Define the objective function to be minimized, according to the formula above. Use $\alpha=10$ (i.e. reaching the target point is quite important).

In [21]:
def objective(thetas, target):
    """
    The objective function to be minimized. It outputs a float as the objective value. 

    Parameters:
    -----------
    thetas: numpy.ndarray of shape (n,)
        An array of current guess for angle theta, containing n>=1 value.

    target: numpy.ndarray (2,)
        A 2-D target position. 
    
    Returns:
    --------
    val: numpy float
        The value of the objective function. 
    """
    val = 10*np.linalg.norm(chain(thetas) - target)**2 + np.sum(thetas**2)
    return val

In [22]:
assert type(objective(np.array([0., 0.]), np.array([0., 1.]))) == np.float64
assert np.allclose(objective(np.array([0., 0.]), np.array([0., 1.])), 50)


In [23]:
def optim1(theta, target):
    return opt.minimize(objective, theta, args=target).x

In [24]:
ik_demo(optim1, 5)

Figure(fig_margin={'top': 60, 'bottom': 60, 'left': 60, 'right': 60}, layout=Layout(height='500px', width='500…

**TODO**: (5 pts) Now try to optimize the following function instead, using the same value of $\alpha$. What behavior do you observe? Note that the optimization seems to run much slower than in the example. Why do you think that is?

$$
\DeclareMathOperator*{\argmin}{argmin}
\argmin_{\theta_1,\ldots,\theta_n} \alpha\cdot\|\vp(\theta_1,\ldots,\theta_n)-\vp_\mathrm{target}\|_2^2 + \sum_{i=1}^n\left|\theta_i\right|
$$


In [25]:
def objective(thetas, target):
    """
    The objective function to be minimized. It outputs a float as the objective value. 

    Parameters:
    -----------
    thetas: numpy.ndarray of shape (n,)
        An array of current guess for angle theta, containing n>=1 value.

    target: numpy.ndarray (2,)
        A 2-D target position. 
    
    Returns:
    --------
    val: numpy float
        The value of the objective function. 
    """
    val = 10*np.linalg.norm(chain(thetas) - target)**2 + np.sum(np.abs(thetas))
    return val

In [26]:
assert type(objective(np.array([0., 0.]), np.array([0., 1.]))) == np.float64
assert np.allclose(objective(np.array([0., 0.]), np.array([0., 1.])), 50)


In [27]:
def optim2(theta, target):
    return opt.minimize(objective, theta, args=target).x

In [28]:
ik_demo(optim2, 5)

Figure(fig_margin={'top': 60, 'bottom': 60, 'left': 60, 'right': 60}, layout=Layout(height='500px', width='500…

# Problem 3: True/False questions (10pts)

Please indicate your answers in the code cell below.

**(1)** In the previous part of the homework, L1 norm regularization runs slower than L2 norm regularization because the L2 norm contains a discontinuity in its derivative. 

**(2)** The Jacobian matrix of a function is a square matrix when the number of input variables is equal to the number of output variables.

**(3)** If a point is a local minimum of a function, then the Hessian of that function is positive-definite at that point.

**(4)** L2 regularization is more likely to encourage all model weights to have small values rather than driving some to zero.

**(5)** L1 regularization encourages sparsity by promoting some model weights to be exactly zero.


In [29]:
# Change these variables to indicate your answer.
A1 = False
A2 = True
A3 = False
A4 = True
A5 = True


In [30]:
answers = [A1, A2, A3, A4, A5]
for answer in answers:
    assert (answer is None) or (answer is True) or (answer is False), "The answers should be True/False or None!"
print("Congratulations! You finished the third assignment of CS328!")

Congratulations! You finished the third assignment of CS328!
